# N125 · 位掩码、异或与二进制Trie

**目标：** 把集合操作和按位最优选择转换成整数操作。

**先修：** N124, N077, N015。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 子集/子掩码；XOR消去；位分治；二进制Trie；线性基扩展。

## 从问题到算法

子掩码枚举中的(s−1)&mask先减一，再清除不属于原集合的位，可按降序遍历所有子集。异或能抵消出现偶数次的数。最大异或Trie按高位优先寻找相反位：高位增加的收益大于所有低位之和，因此可以逐位贪心。

## 最小实现

**本章接口：** `enumerate_submasks(mask)`、`single_number(nums)`、`BinaryTrie`、`find_maximum_xor(nums)`

In [1]:
def enumerate_submasks(mask):
    if mask<0: raise ValueError('nonnegative mask required')
    out=[]; sub=mask
    while True:
        out.append(sub)
        if sub==0: return out
        sub=(sub-1)&mask

def single_number(nums):
    answer=0
    for x in nums: answer^=x
    return answer

class BinaryTrie:
    def __init__(self,bits=32):
        if bits<1: raise ValueError('positive bit width required')
        self.bits=bits; self.children=[[-1,-1]]; self.size=0
    def _check(self,x):
        if not 0<=x<1<<self.bits: raise ValueError('value outside bit width')
    def insert(self,x):
        self._check(x); node=0
        for shift in range(self.bits-1,-1,-1):
            bit=x>>shift&1
            if self.children[node][bit]==-1: self.children[node][bit]=len(self.children); self.children.append([-1,-1])
            node=self.children[node][bit]
        self.size+=1
    def max_xor(self,x):
        self._check(x)
        if not self.size: raise ValueError('empty trie')
        node=answer=0
        for shift in range(self.bits-1,-1,-1):
            bit=x>>shift&1; opposite=bit^1
            if self.children[node][opposite]!=-1: answer|=1<<shift; node=self.children[node][opposite]
            else: node=self.children[node][bit]
        return answer

def find_maximum_xor(nums):
    if not nums: return 0
    trie=BinaryTrie(max(1,max(nums).bit_length()))
    for x in nums: trie.insert(x)
    return max(trie.max_xor(x) for x in nums)

## 正确性、前提与复杂度

子掩码更新给出当前集合编码以下的最大合法编码，因此无重无漏，0必须显式处理否则会回到mask。Trie每一层选可行的最大异或位；若高位可取1，任何高位取0的方案都不可能被低位补偿。

**代价：** 含k个1的掩码有2^k个子集，输出本身就是指数规模。Trie插入/查询O(B)，总O(nB)时间空间上界。最大异或输入为非负整数；单独异或抵消要求除目标外其他元素出现偶数次。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

mask=0b10110
show_table(['子掩码','二进制','元素数'],[(s,format(s,'05b'),s.bit_count()) for s in enumerate_submasks(mask)])
print('最大异或',find_maximum_xor([3,10,5,25,2,8]))

子掩码,二进制,元素数
22,10110,3
20,10100,2
18,10010,2
16,10000,1
6,00110,2
4,00100,1
2,00010,1
0,00000,0


最大异或 28


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
assert single_number([2,2,1])==1
assert find_maximum_xor([3,10,5,25,2,8])==28
for mask in range(64):
    subs=enumerate_submasks(mask)
    assert subs==sorted((s for s in range(mask+1) if s&mask==s),reverse=True)
    assert len(subs)==len(set(subs))==1<<mask.bit_count()
from random import Random
rng=Random(125)
for _ in range(150):
    a=[rng.randrange(1024) for _ in range(rng.randrange(1,16))]
    assert find_maximum_xor(a)==max(x^y for x in a for y in a)
trie=BinaryTrie(4); trie.insert(3); trie.insert(3)
assert trie.max_xor(12)==15
print('N125: 所有本章断言通过')

N125: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 解释遍历所有mask及其submask的总次数。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 扩展可回滚计数的二进制Trie。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 136. Single Number（canonical）
- 260. Single Number III（canonical）
- 421. Maximum XOR of Two Numbers in an Array（canonical）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。